[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_3_5/13_tag_3_5_modelle_speichern_laden.ipynb)

# Tag 3-5 - 13 Modelle speichern und laden

Nach dem Training ist ein Modell erst dann wirklich nützlich, wenn wir es später wiederverwenden können:

- für Inferenz in einem anderen Notebook,
- für Deployment,
- für Vergleichsexperimente,
- oder um den besten Trainingsstand zu sichern.

Dieses Notebook zeigt drei typische Varianten:

1. das finale Modell speichern,
2. während des Trainings automatisch das beste Modell speichern,
3. ein gespeichertes Modell später laden und prüfen.

Wir verwenden das native Keras-Format `.keras`. Es speichert Architektur, Gewichte und Compile-Informationen in einer Datei.


## Lesepfad für dieses Notebook

In diesem Notebook gibt es drei wichtige Namen, die später wieder auftauchen:

- `build_model()`: baut ein frisches Keras-Modell. Die Funktion wird später noch einmal benutzt, wenn wir nur Gewichte laden.
- `best_model_path`: Dateipfad zum besten Modell aus dem Training.
- `final_model_path`: Dateipfad zum letzten Modell nach dem Training.

Wenn eine Funktion oder ein Pfad später wiederverwendet wird, steht im Code jetzt direkt dabei, wofür er gedacht ist.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


## Daten

Wir nehmen wieder ein kleines binäres Klassifikationsproblem. Wichtig ist hier nicht der Datensatz, sondern der Ablauf:

1. Daten vorbereiten,
2. Modell trainieren,
3. besten Stand speichern,
4. Modell neu laden,
5. Vorhersagen vergleichen.


In [ ]:
# X enthält die Eingabemerkmale, y enthält die Zielklasse 0/1.
X, y = make_moons(n_samples=1200, noise=0.25, random_state=RANDOM_STATE)

# StandardScaler ist hier bewusst außerhalb des Modells.
# Das ist später eine wichtige Limitierung: Der Scaler wird nicht automatisch mitgespeichert.
X = StandardScaler().fit_transform(X).astype("float32")
y = y.astype("float32")

# Split in drei Teile:
# - train: Modell lernt daraus.
# - val: ModelCheckpoint entscheidet damit, welches Modell "das beste" ist.
# - test: wird erst ganz am Ende für eine neutrale Prüfung genutzt.
X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.35, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, random_state=RANDOM_STATE, stratify=y_rest
)

pd.DataFrame(X_train, columns=["x1", "x2"]).assign(label=y_train).head()


## Modellfunktion

Eine Modellfunktion ist praktisch, weil wir das gleiche Modell reproduzierbar neu bauen können. Für echtes Laden brauchen wir die Funktion aber nicht: `keras.models.load_model(...)` rekonstruiert die Architektur aus der gespeicherten Datei.


In [ ]:
def build_model():
    # Diese Funktion kapselt nur die Architektur und compile-Einstellungen.
    # Vorteil: Wenn wir später nur Gewichte laden, können wir exakt dieselbe Architektur neu bauen.
    model = keras.Sequential(
        [
            keras.Input(shape=(2,), name="features"),
            layers.Dense(32, activation="relu", name="dense_1"),
            layers.Dense(16, activation="relu", name="dense_2"),
            layers.Dense(1, activation="sigmoid", name="probability"),
        ],
        name="moon_classifier",
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.02),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.BinaryAccuracy(name="accuracy")],
    )
    return model


# Ab hier ist model das konkrete Modellobjekt, das trainiert wird.
model = build_model()
model.summary()


## Bestes Modell automatisch speichern

`ModelCheckpoint` beobachtet eine Metrik, zum Beispiel `val_loss`.

- `save_best_only=True`: Es wird nur gespeichert, wenn sich die überwachte Metrik verbessert.
- `monitor="val_loss"`: Die Validation Loss entscheidet, nicht die Training Loss.
- `mode="min"`: Kleine Validation Loss ist besser.

Das ist meist sinnvoller als blind das letzte Modell zu nehmen. Das letzte Modell kann bereits Overfitting enthalten.


In [ ]:
# In einem normalen Repo-Checkout liegt dieses Notebook unter notebooks/Day_3_5.
# In Colab kann der aktuelle Ordner anders sein. Dann fällt der Code auf "." zurück.
notebook_dir = Path("notebooks/Day_3_5")
if not notebook_dir.exists():
    notebook_dir = Path(".")

# Dieser Ordner ist absichtlich in .gitignore eingetragen.
# Er darf lokal entstehen, soll aber nicht committed werden.
run_dir = notebook_dir / "saved_models" / "13_modelle_speichern_laden"
run_dir.mkdir(parents=True, exist_ok=True)

best_model_path = run_dir / "best_model.keras"
final_model_path = run_dir / "final_model.keras"

# ModelCheckpoint wird später in model.fit(callbacks=[...]) übergeben.
# Während jeder Epoche prüft dieser Callback val_loss und überschreibt best_model.keras nur bei Verbesserung.
checkpoint = callbacks.ModelCheckpoint(
    filepath=best_model_path,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1,
)

early_stopping = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=False,
)

# history enthält nach dem Training die Kurven aus jeder Epoche.
# Der Checkpoint speichert parallel automatisch das beste Modell auf Platte.
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=120,
    batch_size=32,
    callbacks=[checkpoint, early_stopping],
    verbose=0,
)

model.save(final_model_path)

print("Ordner:", run_dir)
print("Bestes Modell:", best_model_path.name, best_model_path.exists())
print("Finales Modell:", final_model_path.name, final_model_path.exists())


In [ ]:
history_df = pd.DataFrame(history.history)
best_epoch = int(history_df["val_loss"].idxmin() + 1)

ax = history_df[["loss", "val_loss"]].plot(figsize=(8, 4))
ax.axvline(best_epoch - 1, color="black", linestyle="--", label=f"beste Epoche: {best_epoch}")
ax.set_title("Training Loss vs. Validation Loss")
ax.set_xlabel("Epoche")
ax.legend()

history_df.iloc[[0, best_epoch - 1, len(history_df) - 1]][["loss", "val_loss", "accuracy", "val_accuracy"]]


## Laden und prüfen

Jetzt simulieren wir ein neues Notebook: Wir nehmen nur den Pfad und laden das Modell aus der Datei.

Ein einfacher Plausibilitätscheck ist: Das geladene Modell sollte auf den gleichen Testdaten dieselben Vorhersagen liefern wie das gespeicherte Modell.


In [ ]:
loaded_best = keras.models.load_model(best_model_path)
loaded_final = keras.models.load_model(final_model_path)

results = pd.DataFrame(
    [
        ("bestes gespeichertes Modell", *loaded_best.evaluate(X_test, y_test, verbose=0)),
        ("finales Modell", *loaded_final.evaluate(X_test, y_test, verbose=0)),
    ],
    columns=["modell", "test_loss", "test_accuracy"],
)
results


In [ ]:
pred_best = loaded_best.predict(X_test[:10], verbose=0)
pred_best_again = keras.models.load_model(best_model_path).predict(X_test[:10], verbose=0)

np.allclose(pred_best, pred_best_again), pred_best[:5].round(3).ravel()


## Nur Gewichte speichern

Manchmal möchte man nur die Gewichte speichern. Dann muss die Architektur beim Laden exakt wieder aufgebaut werden.

Das ist nützlich für Experimente, aber fehleranfälliger als `model.save(...)`, weil Architektur und Compile-Schritt nicht in derselben Datei stecken.


In [ ]:
weights_path = run_dir / "weights_only.weights.h5"
loaded_best.save_weights(weights_path)

same_architecture = build_model()
same_architecture.load_weights(weights_path)

same_architecture.evaluate(X_test, y_test, verbose=0)


## Limitierungen und typische Fehler

- Das Modell speichert nicht automatisch jede externe Vorverarbeitung. Wenn `StandardScaler` außerhalb des Modells genutzt wird, muss auch dieser Scaler gespeichert werden oder die Vorverarbeitung muss ins Keras-Modell integriert werden.
- Bei eigenen Layern, Losses oder Metriken braucht Keras entweder registrierte Klassen/Funktionen oder `custom_objects` beim Laden.
- Ein gutes `val_loss` garantiert nicht automatisch gute Produktionsleistung. Validation-Daten müssen repräsentativ sein.
- `restore_best_weights=True` in `EarlyStopping` stellt Gewichte im aktuellen Python-Objekt wieder her. Ein Checkpoint ist trotzdem nützlich, weil er den Stand dauerhaft auf Platte schreibt.
- Das Speichern des besten Modells ersetzt kein Experiment-Tracking. Lernrate, Datenversion, Seed und Codeversion sollten separat dokumentiert werden.
- Lokal erzeugte Modellartefakte liegen in `notebooks/Day_3_5/saved_models/`. Dieser Ordner ist absichtlich ignoriert, damit große oder versehentlich sensible Dateien nicht in Git landen.
